# Results for model: institute-of-science-tokyo/llama-3.1-swallow-70b-instruct-v0.1

In [ ]:
import polars as pl
import xgboost as xgb
from sklearn.model_selection import train_test_split

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Calculate global TOP_QUANTILE
df = df.with_columns(
    pl.col('feature_00').rank(method='dense').alias('rank')
).filter(
    pl.col('rank') <= 0.15 * df.shape[0]
).drop('rank')

# Convert date_id to datetime
df = df.with_columns(
    pl.col('date_id').str.strptime(pl.Datetime, '%Y%m%d')
)

# Create rolling batches
batches = df.select(
    pl.col('date_id').rolling_batch(
        window_size='1d',
        offset='1d',
        by='1d'
    )
)

# Calculate TOP_QUANTILE for each batch
df = pl.concat(
    [
        batch.with_columns(
            pl.col('feature_00').quantile(0.15).alias('top_quantile')
        )
        for batch in batches
    ]
)

# Train-test split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

# Train XGBoost Regressor
xgb_model = xgb.XGBRegressor()
xgb_model.fit(train_df.drop('responder_6'), train_df['responder_6'])